# Notebook d'exploration 

In [89]:
import pandas as pd
import sys
sys.path.append('../src')
from tool import inspector
import plotly.graph_objects as go

In [90]:
df_customers = pd.read_csv(r"..\data\raw\customers.csv", sep=";")
df_products = pd.read_csv(r"..\data\raw\products.csv", sep=";")
df_transactions = pd.read_csv(r"..\data\raw\Transactions.csv", sep=";")

C:\Users\quent\AppData\Local\Temp\ipykernel_20160\430076701.py:3: DtypeWarning: Columns (0,1,2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_transactions = pd.read_csv(r"..\data\raw\Transactions.csv", sep=";")


In [91]:
inspector(df_customers)

######### DEBUT DE L'ANALYSE #########
Le df comprend 8621 lignes et 3 colonnes 

---------------------------------------------------------------
le df contient les nuls suivant 
client_id    0
sex          0
birth        0
dtype: int64

---------------------------------------------------------------
les types et noms de colonnes sont les suivants
client_id    object
sex          object
birth         int64
dtype: object

---------------------------------------------------------------
le nombre d'uniques dans les colonnes sont les suivants
client_id    8621
sex             2
birth          76
dtype: int64

---------------------------------------------------------------

######### FIN DE L'ANALYSE #########


In [92]:
# fonction permetant de donner le nombres de ligne et de colonnes
print(f"Le df comprend {df_customers.shape[0]} lignes et {df_customers.shape[1]} colonnes ")

Le df comprend 8621 lignes et 3 colonnes 


In [93]:
#fonction comptant le nombre de nulls
count_null = df_customers.isna().sum()
print(f"le df contient les nuls suivant ")
print(count_null)


le df contient les nuls suivant 
client_id    0
sex          0
birth        0
dtype: int64


In [94]:
#fonction donnant le nom et le type de données 
print(f"les types et noms de colonnes sont les suivants")
print(f"{df_customers.dtypes}")

les types et noms de colonnes sont les suivants
client_id    object
sex          object
birth         int64
dtype: object


In [95]:
#fonction donnant le nombre d'unique par colonnes
print("le nombre d'uniques dans les colonnes sont les suivants")
print(f"{df_customers.nunique()}")

le nombre d'uniques dans les colonnes sont les suivants
client_id    8621
sex             2
birth          76
dtype: int64


In [96]:
df_customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8621 entries, 0 to 8620
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   client_id  8621 non-null   object
 1   sex        8621 non-null   object
 2   birth      8621 non-null   int64 
dtypes: int64(1), object(2)
memory usage: 202.2+ KB


In [97]:
df_products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3286 entries, 0 to 3285
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   id_prod  3286 non-null   object 
 1   price    3286 non-null   float64
 2   categ    3286 non-null   int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 77.1+ KB


In [98]:
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   id_prod     687534 non-null  object
 1   date        687534 non-null  object
 2   session_id  687534 non-null  object
 3   client_id   687534 non-null  object
dtypes: object(4)
memory usage: 32.0+ MB


In [99]:
# Supprimer les lignes avec des données manquantes dans toutes les colonnes
df_transactions = df_transactions.dropna()
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
Index: 687534 entries, 0 to 687533
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   id_prod     687534 non-null  object
 1   date        687534 non-null  object
 2   session_id  687534 non-null  object
 3   client_id   687534 non-null  object
dtypes: object(4)
memory usage: 26.2+ MB


In [100]:
df_prod_transac = df_transactions.merge(df_products, on="id_prod", how="left")
print(f" ## merge de df_trasactions et df_products OK ##\n")
print(f" - ligne avant : {len(df_transactions):,}")
print(f" - ligne aprés {len(df_prod_transac):,}")
print(f" - colonne {list(df_prod_transac.columns)}\n")

df = df_prod_transac.merge(df_customers,on="client_id", how="left")
print(f" ## merge de df_prod_trasc et customers OK ##\n")
print(f" - ligne avant : {len(df):,}")
print(f" - ligne aprés {len(df):,}")
print(f" - colonne {list(df.columns)}")


 ## merge de df_trasactions et df_products OK ##

 - ligne avant : 687,534
 - ligne aprés 687,534
 - colonne ['id_prod', 'date', 'session_id', 'client_id', 'price', 'categ']

 ## merge de df_prod_trasc et customers OK ##

 - ligne avant : 687,534
 - ligne aprés 687,534
 - colonne ['id_prod', 'date', 'session_id', 'client_id', 'price', 'categ', 'sex', 'birth']


In [101]:
#convertion des date au bon format
df['date'] = pd.to_datetime(df['date'])

In [102]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 687534 entries, 0 to 687533
Data columns (total 8 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   id_prod     687534 non-null  object        
 1   date        687534 non-null  datetime64[ns]
 2   session_id  687534 non-null  object        
 3   client_id   687534 non-null  object        
 4   price       687534 non-null  float64       
 5   categ       687534 non-null  int64         
 6   sex         687534 non-null  object        
 7   birth       687534 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 42.0+ MB


# Calcul et graphique 
## chiffre d'affaire et moyenne mobile 

In [103]:
#grouper le CA par jour
ca_journalier = df.groupby(pd.Grouper(key='date', freq='D'))['price'].sum().reset_index()
ca_journalier.rename(columns={"price":"ca"}, inplace=True)
ca_journalier.head()

,date,ca
0,2021-03-01,16565.22
1,2021-03-02,15486.45
2,2021-03-03,15198.69
3,2021-03-04,15196.07
4,2021-03-05,17471.37


In [104]:
#trier par date (inspensable pour les moy mobile)
ca_journalier = ca_journalier.sort_values('date').reset_index(drop=True)
#moyenne mobile sur 7 jours
ca_journalier['mm_7j'] = ca_journalier['ca'].rolling(window=7).mean()
#moyenne mobile sur 30 jours 
ca_journalier['mm_30j'] = ca_journalier['ca'].rolling(window=30).mean()

print("moyenne mobiles calculées")
print(ca_journalier.head(10))

moyenne mobiles calculées
        date        ca         mm_7j  mm_30j
0 2021-03-01  16565.22           NaN     NaN
1 2021-03-02  15486.45           NaN     NaN
2 2021-03-03  15198.69           NaN     NaN
3 2021-03-04  15196.07           NaN     NaN
4 2021-03-05  17471.37           NaN     NaN
5 2021-03-06  15785.28           NaN     NaN
6 2021-03-07  14760.20  15780.468571     NaN
7 2021-03-08  15679.53  15653.941429     NaN
8 2021-03-09  15710.51  15685.950000     NaN
9 2021-03-10  15496.87  15728.547143     NaN


In [105]:
#Graphique des moyenne mobile 

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ca_journalier['date'], 
    y=ca_journalier['ca'],
    mode='lines',
    name='CA Journalier',
    line=dict(width=0.3, color='gray') 
))
fig.add_trace(go.Scatter(
    x=ca_journalier['date'], 
    y=ca_journalier['mm_7j'],
    mode='lines',
    name='Moyenne Mobile 7j',
    line=dict(width=3, color='blue') 
))
fig.add_trace(go.Scatter(
    x=ca_journalier['date'], 
    y=ca_journalier['mm_30j'],
    mode='lines',
    name='Moyenne Mobile 30j',
    line=dict(width=4.5, color='red') 
))
# mise en forme
fig.update_layout(
    title="Analyse du CA et Moyennes Mobiles",
    xaxis_title="Date",
    yaxis_title="Montant (€)",
    template="plotly_white"
)

fig.show()

## chiffre d'affaire par catégories 

In [106]:
ca_categorie = df.groupby(by="categ")['price'].sum().reset_index()

ca_categorie.head()

,categ,price
0,0,4419730.97
1,1,4827657.11
2,2,2780275.02


In [107]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=ca_categorie['categ'], 
    y=ca_categorie['price'],
    name='CA par catégories',
    marker_color=['blue', 'green', 'orange']
))
# mise en forme
fig.update_layout(
    title_text='Répartition du CA par catégorie',
    xaxis=dict(type='category')
)

fig.show()

In [108]:
#grouper les clients par mois
ca_mensuel_categ = df.groupby([pd.Grouper(key='date', freq='M'), 'categ'])['price'].sum().reset_index()
ca_mensuel_categ.head()

C:\Users\quent\AppData\Local\Temp\ipykernel_20160\1212606156.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  ca_mensuel_categ = df.groupby([pd.Grouper(key='date', freq='M'), 'categ'])['price'].sum().reset_index()


,date,categ,price
0,2021-03-31,0,193629.17
1,2021-03-31,1,186974.17
2,2021-03-31,2,101837.27
3,2021-04-30,0,205222.46
4,2021-04-30,1,156138.35


In [109]:
#Graphique des moyenne mobile 

fig = go.Figure()


for c in ca_mensuel_categ['categ'].unique():
    # On isole les données de la catégorie
    data_filtree = ca_mensuel_categ[ca_mensuel_categ['categ'] == c]
    
    fig.add_trace(go.Scatter(
        x=data_filtree['date'],
        y=data_filtree['price'],
        mode='lines', 
        name=f"Catégorie {c}",
        connectgaps=True # Au cas où un mois n'aurait pas de données
    ))

# mise en forme
fig.update_layout(
    title="Évolution du CA mensuel par catégorie",
    xaxis_title="Mois",
    yaxis_title="CA (€)",
    hovermode="x unified" # affiche toutes les valeurs au même point X au survol
)

fig.show()

## Nombre de clients par mois 

In [110]:
#grouper les clients par mois
client_mois = df.groupby(pd.Grouper(key='date', freq='M'))['client_id'].nunique().reset_index()
client_mois.rename(columns={"client_id":"client"}, inplace=True)
client_mois.head()

C:\Users\quent\AppData\Local\Temp\ipykernel_20160\91020028.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  client_mois = df.groupby(pd.Grouper(key='date', freq='M'))['client_id'].nunique().reset_index()


,date,client
0,2021-03-31,5676
1,2021-04-30,5674
2,2021-05-31,5644
3,2021-06-30,5659
4,2021-07-31,5672


In [111]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=client_mois['date'], 
    y=client_mois['client'],
    mode='lines',
    name='CA Journalier',
    line=dict(width=2, color='red') 
))
fig.show()

## Nombre de transactions

In [112]:
#grouper les transaction par mois
transac_mois = df.groupby(pd.Grouper(key='date', freq='M'))['session_id'].count().reset_index()
transac_mois.rename(columns={"session_id":"Transaction"}, inplace=True)
transac_mois.head()

C:\Users\quent\AppData\Local\Temp\ipykernel_20160\1900062346.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  transac_mois = df.groupby(pd.Grouper(key='date', freq='M'))['session_id'].count().reset_index()


,date,Transaction
0,2021-03-31,28601
1,2021-04-30,28443
2,2021-05-31,28285
3,2021-06-30,26850
4,2021-07-31,24738


In [113]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=transac_mois['date'], 
    y=transac_mois['Transaction'],
    mode='lines',
    name='Transaction par mois',
    line=dict(width=2, color='red') 
))
fig.show()